# Exercise B — Turn the Alpha Knob into a Scored Sweep
The Pinecone notebook lets you set `alpha` by hand. Here you sweep it and *score* each setting, so the best blend is measured, not guessed.

**Offline scaffold:** we fake dense + sparse scores to show the mechanic. In the real exercise, dense = your embedder, sparse = BM25, scoring = RAGAS.

In [ ]:
# Offline mock so this scaffold runs with NO API key / NO network.
# For real practice, replace `embed()` with your real embedder (InHouseEmbeddings,
# SentenceTransformer, etc.) and `llm()` with a real model call.
import numpy as np, re
_STOP=set("the a an to of and or is are be for in on at by with from as that this it its".split())
def _tok(t): return [w for w in re.findall(r"[a-z0-9]+",t.lower()) if w not in _STOP and len(w)>2]
def embed(texts):
    if isinstance(texts,str): texts=[texts]
    out=[]
    for t in texts:
        v=np.zeros(256)
        for w in _tok(t): v[abs(hash(w))%256]+=1
        n=np.linalg.norm(v); out.append(v/n if n else v)
    return np.array(out)
def cos(a,b): return float(a@b)

# A small corpus standing in for chunks of ERP-2008-chapter4.pdf (health-care economics).
CORPUS = [
 ("Demand for health care is derived from the value of improved health, not the procedures themselves.","demand"),
 ("Health can be defined by longevity (length of life) and quality of life.","demand"),
 ("National health spending reached over 7000 dollars per capita and about 16 percent of GDP.","spending"),
 ("Medical technology accounts for about half of long-term health spending growth.","spending"),
 ("Medicare, enacted in 1965, covers people aged 65 and older; Part D is the drug benefit.","medicare"),
 ("Medicaid, established in 1965, is a program for low-income individuals, administered by states.","medicaid"),
 ("Moral hazard is the tendency to overuse care when insurance covers most of the cost.","moral_hazard"),
 ("Adverse selection is when insurance is most attractive to those most likely to need it.","insurance"),
 ("Health Savings Accounts use pre-tax dollars with high-deductible plans to reduce routine-care reliance.","hsa"),
 ("The proposed standard deduction for health insurance would be a flat 15000 dollars per family.","tax"),
]
texts=[c[0] for c in CORPUS]; sections=[c[1] for c in CORPUS]
print("Mock corpus ready:", len(texts), "chunks.")

In [ ]:
# Simulate dense (meaning) and sparse (keyword) relevance for a few queries.
# Keyword-y queries score higher on sparse; conceptual on dense.
QUERIES = {
    "Medicare Part D":            {"dense": 0.55, "sparse": 0.95},  # exact term -> sparse wins
    "why do health costs rise":   {"dense": 0.90, "sparse": 0.40},  # conceptual -> dense wins
    "HSA pre-tax dollars":        {"dense": 0.65, "sparse": 0.85},
    "value of improved health":   {"dense": 0.88, "sparse": 0.45},
}
def hybrid_score(dense, sparse, alpha):
    return alpha*dense + (1-alpha)*sparse

import numpy as np
alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
print(f"{'query':28s} " + " ".join(f"a={a}" for a in alphas))
for q, s in QUERIES.items():
    scores = [hybrid_score(s["dense"], s["sparse"], a) for a in alphas]
    best_a = alphas[int(np.argmax(scores))]
    print(f"{q:28s} " + " ".join(f"{sc:.2f}" for sc in scores) + f"   best alpha={best_a}")

### Observe & decide
- Keyword queries ('Medicare Part D') peak at low alpha; conceptual queries peak at high alpha. There is no single best alpha — it depends on query mix.
**Your turn:** in the real Pinecone notebook, replace the fake scores with RAGAS faithfulness/answer-relevancy at each alpha, then pick the alpha with the best mean. Tie back to ragas_guide.